# Assignment 1: Project Data Extraction
**Course**: Big Data Analytics (BDA) — 6th Semester  
**Objective**: Compile a dataset of self-reported MDD (Major Depressive Disorder) posts versus Control posts for NLP-based analysis.

### Approach
This notebook demonstrates the entire sequence of data extraction using the official **Reddit API (PRAW)**. It pulls posts from clinical subreddits (`r/depression`, `r/SuicideWatch`) and control subreddits (`r/CasualConversation`). After extraction, it processes and cleans the text to prepare a final structural dataset.

In [ ]:
import praw
import pandas as pd
import numpy as np
import datetime
import re
import nltk
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm

# Ensure NLTK resources are loaded
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

## 1. Authentication
Initialize PRAW using application keys. *(Note: Replace these with your actual credentials when running locally).* 

In [ ]:
reddit = praw.Reddit(
    client_id='YOUR_CLIENT_ID',
    client_secret='YOUR_CLIENT_SECRET',
    user_agent='BDA-MDD-Research/1.0'
)

# Verify connection
print(f"Connected as: {reddit.read_only}")

## 2. Scraping Methodology
We iterate over the target subreddits using PRAW's generator approach.

In [ ]:
def scrape_subreddit(subreddit_name, post_limit, label):
    posts = []
    subreddit = reddit.subreddit(subreddit_name)
    
    print(f"Scraping r/{subreddit_name}...")
    # We iterate over 'new' to avoid bias from 'top' posts.
    for submission in tqdm(subreddit.new(limit=post_limit), total=post_limit):
        # Exclude removed or deleted contents to retain data quality
        if submission.selftext not in ['[removed]', '[deleted]', '']:
            posts.append({
                "post_id": submission.id,
                "subreddit": subreddit_name,
                "timestamp": datetime.datetime.fromtimestamp(submission.created_utc).isoformat(),
                "title": submission.title,
                "selftext": submission.selftext,
                "score": submission.score,
                "num_comments": submission.num_comments,
                "label": label
            })
    return pd.DataFrame(posts)

In [ ]:
# Fetch Class 1 (MDD)
# Due to PRAW limitations we simulate a high limit extraction request (e.g., 2500 each).
df_dep = scrape_subreddit('depression', 2500, 'MDD')
df_sui = scrape_subreddit('SuicideWatch', 2500, 'MDD')

df_mdd = pd.concat([df_dep, df_sui], ignore_index=True)
print(f"Total MDD Extracted: {df_mdd.shape[0]}")

In [ ]:
# Fetch Class 0 (Control)
df_control = scrape_subreddit('CasualConversation', 5000, 'Control')
print(f"Total Control Extracted: {df_control.shape[0]}")

In [ ]:
# Consolidate all rows
df = pd.concat([df_mdd, df_control], ignore_index=True)
# Shuffle the dataset ensuring randomness
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

## 3. NLP Text Cleaning
To prepare for semantic modeling (ClinicalBERT etc.), we apply regex to strip URLs and noisy tags, transform to lowercase, and drop English stop words.

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = [w for w in text.split() if w not in stop_words]
    return " ".join(tokens)

# Hook TQDM to pandas for progress tracking
tqdm.pandas()
df['selftext_cleaned'] = df['selftext'].progress_apply(clean_text)
df['word_count'] = df['selftext_cleaned'].str.split().str.len()

Drop posts that end up being completely empty or structurally insignificant to model on.

In [ ]:
df_clean = df[df['word_count'] >= 5].copy()
print(f"Dropped {df.shape[0] - df_clean.shape[0]} empty/short posts.")
print(f"Final Viable Dataset Size: {df_clean.shape[0]}")

## 4. Feature Engineering: VADER Sentiment Baseline
We assign an initial dimensional polarity score over the origin text to support structural heuristics in later stages.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

df_clean['sentiment_score'] = df_clean.progress_apply(
    lambda row: analyzer.polarity_scores(str(row['selftext']))['compound'], 
    axis=1
)

## 5. Export Secondary Deliverable
Save the cleaned NLP dataset into the environment for ML classification in future workflows.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/reddit_mdd_cleaned.csv', index=False)

df_clean[['subreddit', 'label', 'selftext_cleaned', 'sentiment_score']].head()